**CI twin of `ch06-classification-metrics.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = load_csv("penguins").dropna(subset=["bill_length_mm", "bill_depth_mm"])
y = (df["species"] == "Chinstrap").astype(int)
X = df[["bill_length_mm", "bill_depth_mm"]]
print(f"penguins: {len(y)}, Chinstraps: {y.sum()} ({y.mean():.0%})")

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

lazy = [0] * len(yte)                    # "not a Chinstrap", always
print(f"lazy accuracy:  {accuracy_score(yte, lazy):.3f}")

clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
pred = clf.predict(Xte)
print(f"model accuracy: {accuracy_score(yte, pred):.3f}")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(yte, pred)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"\nTN={tn}  FP={fp}  FN={fn}  TP={tp}")

In [ ]:
from sklearn.metrics import precision_score, recall_score

print(f"precision by hand: {tp / (tp + fp):.3f}")
print(f"recall by hand:    {tp / (tp + fn):.3f}")
print(f"sklearn:           {precision_score(yte, pred):.3f}, "
      f"{recall_score(yte, pred):.3f}")

In [ ]:
probs = clf.predict_proba(Xte)[:, 1]

print("threshold  precision  recall")
for t in (0.2, 0.35, 0.5, 0.65, 0.8):
    p = (probs >= t).astype(int)
    prec = precision_score(yte, p, zero_division=0)
    rec = recall_score(yte, p)
    print(f"   {t:.2f}      {prec:.3f}    {rec:.3f}")

In [ ]:
from sklearn.metrics import f1_score

print(f"F1 at threshold 0.5: {f1_score(yte, pred):.3f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

fpr, tpr, _ = roc_curve(yte, probs)
auc = roc_auc_score(yte, probs)

fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.plot(fpr, tpr, label=f"our model (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], ls="--", lw=1, label="random guessing (AUC = 0.5)")
ax.set_xlabel("false-positive rate")
ax.set_ylabel("true-positive rate")
ax.legend(loc="lower right")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(yte, pred, target_names=["other", "Chinstrap"]))

In [ ]:
def count_cells(y_true, y_pred):
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    return tp, fp, fn, tn

run_tests([
    ("all four piles", count_cells([1, 1, 0, 0, 1, 0], [1, 0, 1, 0, 1, 0]),
     (2, 1, 1, 2)),
    ("no false alarms", count_cells([1, 0, 0], [1, 0, 0]), (1, 0, 0, 2)),
])

In [ ]:
def precision_recall(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return precision, recall

def f1(precision, recall):
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

run_tests([
    ("chapter's matrix", precision_recall(11, 6, 6), (11 / 17, 11 / 17)),
    ("perfect detector", precision_recall(10, 0, 0), (1.0, 1.0)),
    ("lazy baseline", precision_recall(0, 0, 17), (0.0, 0.0)),
    ("f1 of the chapter's pair", round(f1(11 / 17, 11 / 17), 3), 0.647),
    ("f1 punishes lopsidedness", round(f1(1.0, 0.02), 3), 0.039),
    ("f1 of nothing", f1(0.0, 0.0), 0.0),
], tol=1e-9)